In [1]:
import pandas as pd

df = pd.read_csv('../../archive/yellow_tripdata_2015-01.csv')

print(f"Broj redova: {df.shape[0]}")
print(f"Broj kolona: {df.shape[1]}")

Broj redova: 12748986
Broj kolona: 19


In [2]:
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

improvement_surcharge    3
dtype: int64


In [3]:
print("PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)")

zero_distance = df[df['trip_distance'] == 0]
print(f"Vožnje sa 0 milja: {len(zero_distance):,}")

long_distance = df[df['trip_distance'] > 100]
print(f"Vožnje preko 100 milja: {len(long_distance):,}")

zero_fare = df[df['fare_amount'] <= 0]
print(f"Vožnje sa iznosom <= 0: {len(zero_fare):,}")

high_fare = df[df['fare_amount'] > 500]
print(f"Vožnje sa iznosom > 500$: {len(high_fare):,}")

invalid_passengers = df[(df['passenger_count'] == 0) | (df['passenger_count'] > 6)]
print(f"Vožnje sa neispravnim brojem putnika (0 ili >6): {len(invalid_passengers):,}")

negative_tip = df[df['tip_amount'] < 0]
print(f"Vožnje sa negativnom napojnicom: {len(negative_tip):,}")

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]
print(f"Vožnje sa dropoff pre pickup: {len(invalid_time):,}")

PROVERA EKSTREMNIH VREDNOSTI (OUTLIER-A)
Vožnje sa 0 milja: 79,365
Vožnje preko 100 milja: 172
Vožnje sa iznosom <= 0: 7,666
Vožnje sa iznosom > 500$: 77
Vožnje sa neispravnim brojem putnika (0 ili >6): 6,595
Vožnje sa negativnom napojnicom: 79
Vožnje sa dropoff pre pickup: 297


In [4]:

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

print("PROVERA NEKONZISTENTNIH ZAPISA")

invalid_time = df[df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']]
print(f"Vožnje sa dropoff pre pickup: {len(invalid_time)}")

df['duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
long_rides = df[df['duration_minutes'] > 1440]  # 24h = 1440 min
print(f"Vožnje duže od 24h: {len(long_rides)}")

zero_duration = df[df['duration_minutes'] == 0]
print(f"Vožnje koje traju 0 minuta: {len(zero_duration)}")

PROVERA NEKONZISTENTNIH ZAPISA
Vožnje sa dropoff pre pickup: 297
Vožnje duže od 24h: 79
Vožnje koje traju 0 minuta: 14816


In [8]:
import pandas as pd
from sqlalchemy import create_engine
import pyodbc


In [9]:
server = 'DESKTOP-64J7LMV\\MSSQLSERVER2022' 
database = 'SkladištenjePodataka'           

engine = create_engine(
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes'
)

try:
    with engine.connect() as conn:
        print("Povezano!")
except Exception as e:
    print(f"Greška! {e}")

Povezano!
